### This notbook takes in the data in the minimodel form and translates the data into the experanto format

In [1]:
import os
import torch
import numpy as np
from minimodel import data
import yaml
import hashlib
import base64

device = torch.device('cuda')

In [2]:
mouse_id = 0

data_path = '../data'
output_path = './data_experanto'
np.random.seed(1)

In [3]:
# load images
img = data.load_images(data_path, mouse_id, file=data.img_file_name[mouse_id])

# load neurons
fname = '%s_nat60k_%s.npz'%(data.db[mouse_id]['mname'], data.db[mouse_id]['datexp'])
spks, istim_train, istim_test, xpos, ypos, spks_rep_all = data.load_neurons(file_path = os.path.join(data_path, fname), mouse_id = mouse_id)
n_stim, n_max_neurons = spks.shape


# split train and validation set
itrain, ival = data.split_train_val(istim_train, train_frac=0.9)

ineur = np.arange(0, n_max_neurons)
spks_train = spks[itrain][:,ineur]
spks_val = spks[ival][:,ineur]

print(type(istim_train), ' istim_train: ', istim_train.shape, istim_train.min(), istim_train.max())
print(type(itrain), ' itrain: ', itrain.shape, itrain.min(), itrain.max())
print(type(ival), ' ival: ', ival.shape, ival.min(), ival.max())
print(type(spks_train), ' spks_train: ', spks_train.shape, spks_train.min(), spks_train.max())
print(type(spks_val), ' spks_val: ', spks_val.shape, spks_val.min(), spks_val.max())
print()

img_train = img[istim_train][itrain]
img_val = img[istim_train][ival]
img_test = img[istim_test]

print(type(img_train), ' img_train: ', img_train.shape, img_train.min(), img_train.max())
print(type(img_val), ' img_val: ', img_val.shape, img_val.min(), img_val.max())
print(type(img_test), ' img_test: ', img_test.shape, img_test.min(), img_test.max())

input_Ly, input_Lx = img_train.shape[-2:]
print("input_Ly: ", input_Ly, " | input_Lx: ", input_Lx)

raw image shape:  (68000, 66, 264)
cropped image shape:  (68000, 66, 130)
img:  (68000, 66, 130) -2.062947 2.088608 float32

loading activities from ../data/L1_A5_nat60k_2023_02_27.npz

splitting training and validation set...
itrain:  (24779,)
ival:  (2754,)
<class 'numpy.ndarray'>  istim_train:  (27533,) 501 59994
<class 'numpy.ndarray'>  itrain:  (24779,) 1 27532
<class 'numpy.ndarray'>  ival:  (2754,) 0 27530
<class 'numpy.ndarray'>  spks_train:  (24779, 6636) 0.0 18.46728334248839
<class 'numpy.ndarray'>  spks_val:  (2754, 6636) 0.0 15.588095449232007

<class 'numpy.ndarray'>  img_train:  (24779, 66, 130) -2.062947 2.088608
<class 'numpy.ndarray'>  img_val:  (2754, 66, 130) -2.062947 2.088608
<class 'numpy.ndarray'>  img_test:  (500, 66, 130) -2.062947 2.088608
input_Ly:  66  | input_Lx:  130


In [4]:
# Initialize lists to store reshaped spks_test and img_test
spks_test_list = []
img_test_list = []
image_id_test_list = []

# Iterate over each stimulus in the test set
for i, spks in enumerate(spks_rep_all):
    nrep, nneurons = spks.shape
    spks_test_list.append(spks)
    
    # Reshape the corresponding img_test
    img_rep = img_test[i][None, :, :].repeat(nrep, axis=0)
    img_test_list.append(img_rep)
    
    # Repeat the image ID for each repeat
    image_id_test_list.append(np.repeat(istim_test[i], nrep))

# Concatenate the reshaped test data
spks_test = np.concatenate(spks_test_list, axis=0)
img_test = np.concatenate(img_test_list, axis=0)
image_id_test = np.concatenate(image_id_test_list, axis=0)

print('spks_test: ', spks_test.shape, spks_test.min(), spks_test.max())
print('img_test: ', img_test.shape, img_test.min(), img_test.max())
print('image_id_test: ', image_id_test.shape, image_id_test.min(), image_id_test.max())

# Concatenate train, val, and test sets
spk_all = np.concatenate([spks_train, spks_val, spks_test], axis=0)
img_all = np.concatenate([img_train, img_val, img_test], axis=0)
image_id_all = np.concatenate([istim_train[itrain], istim_train[ival], image_id_test], axis=0)

# setting up the tiers
NT, NN = spk_all.shape
ntrain, nval, ntest = len(spks_train), len(spks_val), len(spks_test)
itrain = np.arange(ntrain)
ival = np.arange(ntrain, ntrain + nval)
itest = np.arange(ntrain + nval, ntrain + nval + ntest)

tiers = np.zeros(NT, object)
tiers[itrain] = 'train'
tiers[ival] = 'validation'
tiers[itest] = 'test'

print(type(spk_all), ' spks_all: ', spk_all.shape, spk_all.min(), spk_all.max())
print(type(img_all), ' img_all: ', img_all.shape, img_all.min(), img_all.max())
print(type(image_id_all), ' image_id_all: ', image_id_all.shape, image_id_all.min(), image_id_all.max())

spks_test:  (4907, 6636) -2.220446e-16 14.66146
img_test:  (4907, 66, 130) -2.062947 2.088608
image_id_test:  (4907,) 0 499
<class 'numpy.ndarray'>  spks_all:  (32440, 6636) -2.220446049250313e-16 18.46728334248839
<class 'numpy.ndarray'>  img_all:  (32440, 66, 130) -2.062947 2.088608
<class 'numpy.ndarray'>  image_id_all:  (32440,) 0 59994


In [5]:
# make folders
output_folder = f'./nat30k_{data.mouse_names[mouse_id]}_{data.exp_date[mouse_id]}_experanto'
output_root = os.path.join(output_path, output_folder)
os.makedirs(data_path, exist_ok=True)

# set up data folder
img_path = os.path.join(output_root, 'screen')
spk_path = os.path.join(output_root, 'responses')
os.makedirs(img_path, exist_ok=True)
os.makedirs(spk_path, exist_ok=True)

# set up screen
screen_data_path = os.path.join(img_path, 'data')
screen_meta_path = os.path.join(img_path, 'meta')
os.makedirs(screen_data_path, exist_ok=True)
os.makedirs(screen_meta_path, exist_ok=True)

In [6]:
# compute hashes
hashes = []
base_hasher = hashlib.blake2b(digest_size=16)

for b in image_id_all.view(np.uint8).reshape(-1, 8):
    h = base_hasher.copy()
    h.update(b)
    hashes.append(base64.b64encode(h.digest()).decode()[:20])


In [7]:
# I dont know in which order the images were presented. There is no temporal connection! Values are made up.
# Should still work, since they already interpolated the data.

# save screen
# save images and meta data for images
image_size = [int(input_Ly), int(input_Lx)]

for i in range(img_all.shape[0]):
    img_savepath = os.path.join(screen_data_path, f'{i:06d}.npy')
    meta_savepath = os.path.join(screen_meta_path, f'{i:06d}.yml')

    timg = img_all[i]
    np.save(img_savepath, timg[np.newaxis, ...])


    data = {
        "condition_hash": hashes[i],
        "first_frame_idx": i,
        "image_class": "natural greyscale image (from minimodel)",
        "image_id": int(image_id_all[i]),
        "image_size": image_size,
        "modality": "image",
        "num_frames": 1,
        "pre_blank_period": 0.0667,         # 66.7 ms
        "presentation_time": 0.0667,
        "stim_type": "stimulus.Frame",
        "tier": str(tiers[i]),
        "trial_idx": i,
    }
    
    with open(meta_savepath, "w") as f:
        yaml.safe_dump(data, f)


